# Notebook 01 - Setup & Environment
### Spain Tactical Analysis: Qatar 2022 - Euro 2024

**Purpose:** Verify that every required library is installed and working, confirm access to StatsBomb Open Data, and identify the exact competition/season IDs we need for all downstream notebooks.

**Outputs:**
- Printed version table for all libraries
- Full competition list from StatsBomb
- Confirmed IDs for FIFA World Cup 2022 and UEFA Euro 2024
- A match-level inventory table for Spain in both tournaments
- Saved inventory CSV (`outputs/data/spain_match_inventory.csv`)
- Config file (`utils/config.py`)

---
## Section 1 - Library Version Check

In [14]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import mplsoccer
import scipy
import statsbombpy

print("=" * 45)
print("  ENVIRONMENT CHECK - LIBRARY VERSIONS")
print("=" * 45)
print(f"  Python        : {sys.version.split()[0]}")
print(f"  pandas        : {pd.__version__}")
print(f"  numpy         : {np.__version__}")
print(f"  matplotlib    : {matplotlib.__version__}")
print(f"  mplsoccer     : {mplsoccer.__version__}")
print(f"  scipy         : {scipy.__version__}")
print(f"  statsbombpy   : installed")
print("=" * 45)
print("  All libraries imported successfully.")
print("=" * 45)

  ENVIRONMENT CHECK - LIBRARY VERSIONS
  Python        : 3.11.9
  pandas        : 3.0.3
  numpy         : 2.4.6
  matplotlib    : 3.11.0
  mplsoccer     : 1.6.1
  scipy         : 1.17.1
  statsbombpy   : installed
  All libraries imported successfully.


---
## Section 2 - Browse All Available StatsBomb Open Data Competitions

In [15]:
from statsbombpy import sb

all_competitions = sb.competitions()

print(f"Total competitions available: {len(all_competitions)}\n")

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

display(all_competitions[
    ['competition_id', 'season_id', 'competition_name', 'season_name', 'country_name']
].sort_values(['competition_name', 'season_name']))

Total competitions available: 80



,competition_id,season_id,competition_name,season_name,country_name
1,9,27,1. Bundesliga,2015/2016,Germany
0,9,281,1. Bundesliga,2023/2024,Germany
2,1267,107,African Cup of Nations,2023,Africa
20,16,276,Champions League,1970/1971,Europe
19,16,71,Champions League,1971/1972,Europe
18,16,277,Champions League,1972/1973,Europe
17,16,76,Champions League,1999/2000,Europe
16,16,44,Champions League,2003/2004,Europe
15,16,37,Champions League,2004/2005,Europe
14,16,39,Champions League,2006/2007,Europe


---
## Section 3 - Identify Our Target Competitions

In [16]:
wc2022 = all_competitions[
    all_competitions['competition_name'].str.contains('World Cup', case=False) &
    all_competitions['season_name'].str.contains('2022', case=False)
]

euro2024 = all_competitions[
    all_competitions['competition_name'].str.contains('Euro', case=False) &
    all_competitions['season_name'].str.contains('2024', case=False)
]

print("FIFA World Cup 2022 entry:")
print(wc2022[['competition_id', 'season_id', 'competition_name', 'season_name']].to_string(index=False))

print("\nUEFA Euro 2024 entry:")
print(euro2024[['competition_id', 'season_id', 'competition_name', 'season_name']].to_string(index=False))

FIFA World Cup 2022 entry:
 competition_id  season_id competition_name season_name
             43        106   FIFA World Cup        2022

UEFA Euro 2024 entry:
 competition_id  season_id competition_name season_name
             55        282        UEFA Euro        2024


In [17]:
assert len(wc2022) > 0, "World Cup 2022 NOT found in StatsBomb Open Data!"
assert len(euro2024) > 0, "Euro 2024 NOT found in StatsBomb Open Data!"

WC2022_COMPETITION_ID = int(wc2022['competition_id'].iloc[0])
WC2022_SEASON_ID      = int(wc2022['season_id'].iloc[0])

EURO2024_COMPETITION_ID = int(euro2024['competition_id'].iloc[0])
EURO2024_SEASON_ID      = int(euro2024['season_id'].iloc[0])

print("=" * 50)
print("  TARGET COMPETITION IDs")
print("=" * 50)
print(f"  FIFA World Cup 2022")
print(f"    competition_id : {WC2022_COMPETITION_ID}")
print(f"    season_id      : {WC2022_SEASON_ID}")
print()
print(f"  UEFA Euro 2024")
print(f"    competition_id : {EURO2024_COMPETITION_ID}")
print(f"    season_id      : {EURO2024_SEASON_ID}")
print("=" * 50)
print("  Both competitions confirmed in Open Data.")
print("=" * 50)

  TARGET COMPETITION IDs
  FIFA World Cup 2022
    competition_id : 43
    season_id      : 106

  UEFA Euro 2024
    competition_id : 55
    season_id      : 282
  Both competitions confirmed in Open Data.


---
## Section 4 - Pull All Matches for Both Tournaments

In [18]:
wc2022_all_matches   = sb.matches(competition_id=WC2022_COMPETITION_ID,   season_id=WC2022_SEASON_ID)
euro2024_all_matches = sb.matches(competition_id=EURO2024_COMPETITION_ID, season_id=EURO2024_SEASON_ID)

print(f"World Cup 2022 - total matches in dataset : {len(wc2022_all_matches)}")
print(f"Euro 2024      - total matches in dataset : {len(euro2024_all_matches)}")

World Cup 2022 - total matches in dataset : 64
Euro 2024      - total matches in dataset : 51


In [19]:
def filter_spain_matches(matches_df):
    """Return only rows where Spain is the home or away team."""
    spain = matches_df[
        (matches_df['home_team'] == 'Spain') |
        (matches_df['away_team'] == 'Spain')
    ].copy()

    spain['opponent'] = spain.apply(
        lambda r: r['away_team'] if r['home_team'] == 'Spain' else r['home_team'], axis=1
    )
    spain['spain_score'] = spain.apply(
        lambda r: r['home_score'] if r['home_team'] == 'Spain' else r['away_score'], axis=1
    )
    spain['opp_score'] = spain.apply(
        lambda r: r['away_score'] if r['home_team'] == 'Spain' else r['home_score'], axis=1
    )
    spain['result'] = spain.apply(
        lambda r: 'W' if r['spain_score'] > r['opp_score']
                  else ('L' if r['spain_score'] < r['opp_score'] else 'D'), axis=1
    )
    return spain.sort_values('match_date').reset_index(drop=True)


spain_wc2022   = filter_spain_matches(wc2022_all_matches)
spain_euro2024 = filter_spain_matches(euro2024_all_matches)

print(f"Spain matches - World Cup 2022 : {len(spain_wc2022)}")
print(f"Spain matches - Euro 2024      : {len(spain_euro2024)}")

Spain matches - World Cup 2022 : 4
Spain matches - Euro 2024      : 7


---
## Section 5 - Match Inventory Tables

In [20]:
stage_col = 'competition_stage' if 'competition_stage' in spain_wc2022.columns else 'stage'

print("=" * 65)
print("  SPAIN - FIFA WORLD CUP 2022")
print("=" * 65)
display(
    spain_wc2022[['match_id', 'match_date', stage_col, 'opponent',
                  'spain_score', 'opp_score', 'result']]
)

print("\n")
print("=" * 65)
print("  SPAIN - UEFA EURO 2024")
print("=" * 65)
display(
    spain_euro2024[['match_id', 'match_date', stage_col, 'opponent',
                    'spain_score', 'opp_score', 'result']]
)

  SPAIN - FIFA WORLD CUP 2022


,match_id,match_date,competition_stage,opponent,spain_score,opp_score,result
0,3857291,2022-11-23,Group Stage,Costa Rica,7,0,W
1,3857263,2022-11-27,Group Stage,Germany,1,1,D
2,3857255,2022-12-01,Group Stage,Japan,1,2,L
3,3869220,2022-12-06,Round of 16,Morocco,0,0,D




  SPAIN - UEFA EURO 2024


,match_id,match_date,competition_stage,opponent,spain_score,opp_score,result
0,3930160,2024-06-15,Group Stage,Croatia,3,0,W
1,3930172,2024-06-20,Group Stage,Italy,1,0,W
2,3930179,2024-06-24,Group Stage,Albania,1,0,W
3,3941018,2024-06-30,Round of 16,Georgia,4,1,W
4,3942226,2024-07-05,Quarter-finals,Germany,2,1,W
5,3942752,2024-07-09,Semi-finals,France,2,1,W
6,3943043,2024-07-14,Final,England,2,1,W


---
## Section 6 - Check 360 / Freeze-Frame Data Availability

In [21]:
def check_360_availability(match_id, label):
    """Try to load 360 data for a given match_id."""
    try:
        frames = sb.frames(match_id=match_id)
        if frames is not None and len(frames) > 0:
            print(f"  [OK]  {label} (match {match_id}) -- 360 data available  ({len(frames):,} frames)")
            return True
        else:
            print(f"  [--]  {label} (match {match_id}) -- 360 data is EMPTY")
            return False
    except Exception as e:
        print(f"  [ERR] {label} (match {match_id}) -- 360 data NOT available ({e})")
        return False


print("Checking 360 data availability...\n")
wc_360   = check_360_availability(spain_wc2022['match_id'].iloc[0],   "World Cup 2022 - first Spain match")
euro_360 = check_360_availability(spain_euro2024['match_id'].iloc[0], "Euro 2024      - first Spain match")

print()
if not wc_360:
    print("  NOTE: World Cup 2022 -- We will rely on standard event data only.")
if not euro_360:
    print("  NOTE: Euro 2024 -- We will rely on standard event data only.")

Checking 360 data availability...

  [ERR] World Cup 2022 - first Spain match (match 3857291) -- 360 data NOT available (Reindexing only valid with uniquely valued Index objects)
  [ERR] Euro 2024      - first Spain match (match 3930160) -- 360 data NOT available (Reindexing only valid with uniquely valued Index objects)

  NOTE: World Cup 2022 -- We will rely on standard event data only.
  NOTE: Euro 2024 -- We will rely on standard event data only.


---
## Section 7 - Quick Sanity Check on Event Data

In [22]:
cr_mask = spain_wc2022['opponent'].str.contains('Costa Rica', case=False, na=False)
if cr_mask.any():
    sample_match = spain_wc2022.loc[cr_mask, 'match_id'].iloc[0]
    sample_label = "Spain vs Costa Rica (World Cup 2022 Group Stage)"
else:
    sample_match = spain_wc2022['match_id'].iloc[0]
    sample_label = f"Spain vs {spain_wc2022['opponent'].iloc[0]} (World Cup 2022)"

print(f"Loading sample events: {sample_label}")
print(f"match_id = {sample_match}")

events = sb.events(match_id=sample_match)

print(f"\nTotal events in match : {len(events):,}")
print(f"Columns               : {list(events.columns[:10])} ... (and more)")
print(f"\nEvent types present:")
print(events['type'].value_counts().head(15))

Loading sample events: Spain vs Costa Rica (World Cup 2022 Group Stage)
match_id = 3857291

Total events in match : 4,351
Columns               : ['ball_receipt_outcome', 'ball_recovery_offensive', 'ball_recovery_recovery_failure', 'block_deflection', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot'] ... (and more)

Event types present:
type
Pass              1340
Ball Receipt*     1247
Carry             1111
Pressure           229
Ball Recovery       74
Duel                67
Block               34
Dispossessed        28
Miscontrol          28
Clearance           26
Dribble             25
Goal Keeper         24
Foul Committed      22
Foul Won            20
Shot                17
Name: count, dtype: int64


In [23]:
spain_passes = events[
    (events['type'] == 'Pass') &
    (events['team'] == 'Spain')
].head(5)

print("Sample Spain pass events (first 5):")
display(spain_passes[['minute', 'player', 'type', 'pass_length', 'pass_angle', 'location']].reset_index(drop=True))

Sample Spain pass events (first 5):


,minute,player,type,pass_length,pass_angle,location
0,0,Pablo Martín Páez Gavira,Pass,26.226894,-2.922510,"[61.0, 40.1]"
1,0,Aymeric Laporte,Pass,68.275400,0.533135,"[40.5, 34.8]"
2,0,César Azpilicueta Tanco,Pass,28.939419,-2.915053,"[74.5, 80.0]"
3,0,Rodrigo Hernández Cascante,Pass,27.521992,-1.610775,"[39.7, 69.8]"
4,0,Aymeric Laporte,Pass,18.250753,-1.129204,"[50.0, 29.5]"


---
## Section 8 - Save Match Inventory to Disk

In [24]:
import os

os.makedirs('../outputs/data', exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

spain_wc2022_save   = spain_wc2022.copy()
spain_euro2024_save = spain_euro2024.copy()

spain_wc2022_save['tournament']   = 'WC2022'
spain_euro2024_save['tournament'] = 'EURO2024'

inventory = pd.concat([spain_wc2022_save, spain_euro2024_save], ignore_index=True)

inventory_path = '../outputs/data/spain_match_inventory.csv'
inventory.to_csv(inventory_path, index=False)

print(f"Match inventory saved to: {inventory_path}")
print(f"    Rows: {len(inventory)} ({len(spain_wc2022)} WC2022 + {len(spain_euro2024)} EURO2024)")

Match inventory saved to: ../outputs/data/spain_match_inventory.csv
    Rows: 11 (4 WC2022 + 7 EURO2024)


---
## Section 9 - Save Competition IDs to a Config File

In [25]:
os.makedirs('../utils', exist_ok=True)

config_content = (
    '# utils/config.py\n'
    '# Auto-generated by 01_setup_and_environment.ipynb\n'
    '# These IDs are confirmed against StatsBomb Open Data.\n'
    '\n'
    '# Competition IDs\n'
    f'WC2022_COMPETITION_ID   = {WC2022_COMPETITION_ID}\n'
    f'WC2022_SEASON_ID        = {WC2022_SEASON_ID}\n'
    '\n'
    f'EURO2024_COMPETITION_ID = {EURO2024_COMPETITION_ID}\n'
    f'EURO2024_SEASON_ID      = {EURO2024_SEASON_ID}\n'
    '\n'
    '# Paths\n'
    'OUTPUTS_DATA_DIR    = "../outputs/data"\n'
    'OUTPUTS_FIGURES_DIR = "../outputs/figures"\n'
    '\n'
    '# Tournament labels (for plots and file names)\n'
    'TOURNAMENT_LABELS = {\n'
    '    "WC2022"   : "FIFA World Cup 2022",\n'
    '    "EURO2024" : "UEFA Euro 2024",\n'
    '}\n'
)

config_path = '../utils/config.py'
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_content)

with open('../utils/__init__.py', 'w', encoding='utf-8') as f:
    f.write('')

print(f"Config saved to: {config_path}")
print()
print("Contents:")
print(config_content)

Config saved to: ../utils/config.py

Contents:
# utils/config.py
# Auto-generated by 01_setup_and_environment.ipynb
# These IDs are confirmed against StatsBomb Open Data.

# Competition IDs
WC2022_COMPETITION_ID   = 43
WC2022_SEASON_ID        = 106

EURO2024_COMPETITION_ID = 55
EURO2024_SEASON_ID      = 282

# Paths
OUTPUTS_DATA_DIR    = "../outputs/data"
OUTPUTS_FIGURES_DIR = "../outputs/figures"

# Tournament labels (for plots and file names)
TOURNAMENT_LABELS = {
    "WC2022"   : "FIFA World Cup 2022",
    "EURO2024" : "UEFA Euro 2024",
}



---
## Section 10 - Summary

In [26]:
print("=" * 60)
print("  NOTEBOOK 01 -- COMPLETE")
print("=" * 60)
print("  [OK] All libraries imported and version-checked")
print(f"  [OK] World Cup 2022   -> competition_id={WC2022_COMPETITION_ID}, season_id={WC2022_SEASON_ID}")
print(f"  [OK] Euro 2024        -> competition_id={EURO2024_COMPETITION_ID}, season_id={EURO2024_SEASON_ID}")
print(f"  [OK] Spain WC 2022    -> {len(spain_wc2022)} matches")
print(f"  [OK] Spain Euro 2024  -> {len(spain_euro2024)} matches")
print(f"  [OK] Match inventory  -> saved to outputs/data/spain_match_inventory.csv")
print(f"  [OK] Config file      -> saved to utils/config.py")
print()
print("  NEXT: Notebook 02 -- Load & Prepare Data")
print("=" * 60)

  NOTEBOOK 01 -- COMPLETE
  [OK] All libraries imported and version-checked
  [OK] World Cup 2022   -> competition_id=43, season_id=106
  [OK] Euro 2024        -> competition_id=55, season_id=282
  [OK] Spain WC 2022    -> 4 matches
  [OK] Spain Euro 2024  -> 7 matches
  [OK] Match inventory  -> saved to outputs/data/spain_match_inventory.csv
  [OK] Config file      -> saved to utils/config.py

  NEXT: Notebook 02 -- Load & Prepare Data
